# StyleGAN Architecture for Fashion-MNIST Synthesis

**Author**: Eshan M (#E0324009)  
**Implementation**: PyTorch implementation of StyleGAN (Karras et al.) featuring Mapping Network ($z \to w$), Adaptive Instance Normalization (AdaIN), Progressive Synthesis Blocks, Stochastic Noise Injection, Style Mixing Regularization, and Minibatch Standard Deviation.

---
## 1. Environment Setup & Seed Initialization

In [1]:
# #E0324009 - Eshan M
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
import torchvision
import torchvision.transforms as transforms
import matplotlib.pyplot as plt
import numpy as np

# Set random seed for reproducibility
torch.manual_seed(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using execution device environment: {device}")

Using execution device environment: cuda


## 2. Dataset Pipeline & Transformation Scale

In [2]:
# Create image loading transformations tailored to StyleGAN dimensions
transform = transforms.Compose([
    transforms.Resize((32, 32)),
    transforms.ToTensor(),
    transforms.Lambda(lambda x: x.repeat(3, 1, 1)), # Tile 1-channel to 3 channels (RGB simulation)
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5)) # Standard normalisation scale [-1, 1]
])

batch_size = 64
dataset = torchvision.datasets.FashionMNIST(root='./data', train=True, download=True, transform=transform)
dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=True, drop_last=True)
print(f"Dataset completely initialized. Batches available: {len(dataloader)}")

Dataset completely initialized. Batches available: 937


## 3. Core StyleGAN Components: Mapping Network & AdaIN

In [3]:
class MappingNetwork(nn.Module):
    """Maps random noise vector z to disentangled intermediate latent space w via a robust 4-layer MLP."""
    def __init__(self, latent_dim=128):
        super(MappingNetwork, self).__init__()
        layers = []
        # Reduced from 8 layers to 4 layers to avoid premature representational collapse
        for _ in range(4):
            layers.append(nn.Linear(latent_dim, latent_dim))
            layers.append(nn.LeakyReLU(0.2))
        self.mapping = nn.Sequential(*layers)

    def forward(self, z):
        return self.mapping(z)

class AdaIN(nn.Module):
    """Adaptive Instance Normalization Layer scaling and shifts channels using style vectors."""
    def __init__(self, channels, latent_dim=128):
        super(AdaIN, self).__init__()
        self.instance_norm = nn.InstanceNorm2d(channels, affine=False)
        self.style_scale = nn.Linear(latent_dim, channels)
        self.style_shift = nn.Linear(latent_dim, channels)

    def forward(self, x, w):
        x_norm = self.instance_norm(x)
        gamma = self.style_scale(w).unsqueeze(-1).unsqueeze(-1)
        beta = self.style_shift(w).unsqueeze(-1).unsqueeze(-1)
        return gamma * x_norm + beta

## 4. Synthesis Generator Architecture

In [4]:
class StyleConvBlock(nn.Module):
    """Standard Convolutional step embedded with independent AdaIN layers for progressive style injection."""
    def __init__(self, in_channels, out_channels, latent_dim=128):
        super(StyleConvBlock, self).__init__()
        self.conv = nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1)
        self.adain = AdaIN(out_channels, latent_dim)
        self.act = nn.LeakyReLU(0.2)

    def forward(self, x, w):
        x = self.conv(x)
        x = self.act(x)
        x = self.adain(x, w)
        return x

class StyleGenerator(nn.Module):
    """Synthesis Network starting from a learned constant tensor modulated progressively."""
    def __init__(self, latent_dim=128):
        super(StyleGenerator, self).__init__()
        self.mapping = MappingNetwork(latent_dim)
        
        # Initialize structural constant base tensor as ones so it is highly modifiable
        self.const = nn.Parameter(torch.ones(1, 128, 4, 4))
        
        # Progressive synthesis scales
        self.block1 = StyleConvBlock(128, 128, latent_dim)                           # Scale 4x4
        self.upsample1 = nn.Upsample(scale_factor=2, mode='bilinear', align_corners=False) # to 8x8
        self.block2 = StyleConvBlock(128, 64, latent_dim)
        self.upsample2 = nn.Upsample(scale_factor=2, mode='bilinear', align_corners=False) # to 16x16
        self.block3 = StyleConvBlock(64, 32, latent_dim)
        self.upsample3 = nn.Upsample(scale_factor=2, mode='bilinear', align_corners=False) # to 32x32
        self.block4 = StyleConvBlock(32, 3, latent_dim)
        self.to_rgb = nn.Tanh()

    def forward(self, z, style_mixing_step=None, z2=None):
        w = self.mapping(z)
        w2 = self.mapping(z2) if z2 is not None else w
        
        x = self.const.repeat(z.size(0), 1, 1, 1)
        
        # Add small spatial noise for stochastic variation to break mode collapse
        noise1 = torch.randn(x.size(0), 1, x.size(2), x.size(3), device=x.device) * 0.15
        x = x + noise1
        
        # Style Injection selection parameters
        w_early = w
        w_late = w2 if (style_mixing_step is not None and style_mixing_step <= 2) else w
        
        x = self.block1(x, w_early)
        x = self.upsample1(x)
        
        noise2 = torch.randn(x.size(0), 1, x.size(2), x.size(3), device=x.device) * 0.15
        x = x + noise2
        x = self.block2(x, w_early)
        
        w_late = w2 if (style_mixing_step is not None and style_mixing_step <= 4) else w
        x = self.upsample2(x)
        
        noise3 = torch.randn(x.size(0), 1, x.size(2), x.size(3), device=x.device) * 0.15
        x = x + noise3
        x = self.block3(x, w_late)
        x = self.upsample3(x)
        
        noise4 = torch.randn(x.size(0), 1, x.size(2), x.size(3), device=x.device) * 0.15
        x = x + noise4
        x = self.block4(x, w_late)
        
        return self.to_rgb(x)

## 5. Discriminator Architecture with Minibatch Standard Deviation

In [5]:
class MinibatchStdDev(nn.Module):
    """Appends standard deviation of the batch across channels to force batch diversity."""
    def forward(self, x):
        batch_size, _, height, width = x.shape
        # Compute average standard deviation over all features and pixels
        std = torch.std(x, dim=0, keepdim=True) + 1e-8
        mean_std = torch.mean(std).view(1, 1, 1, 1)
        # Tile and append as a new channel
        mean_std = mean_std.repeat(batch_size, 1, height, width)
        return torch.cat([x, mean_std], dim=1)

class Discriminator(nn.Module):
    """Standard CNN classification network verifying macro structural realness frames with Batch Diversity check."""
    def __init__(self):
        super(Discriminator, self).__init__()
        self.model = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=4, stride=2, padding=1), # 32x32 -> 16x16
            nn.LeakyReLU(0.2),
            nn.Conv2d(32, 64, kernel_size=4, stride=2, padding=1), # 16x16 -> 8x8
            nn.BatchNorm2d(64),
            nn.LeakyReLU(0.2),
            nn.Conv2d(64, 128, kernel_size=4, stride=2, padding=1), # 8x8 -> 4x4
            nn.BatchNorm2d(128),
            nn.LeakyReLU(0.2),
            MinibatchStdDev(), # Appends diversity statistic
            nn.Conv2d(129, 128, kernel_size=3, padding=1), # process the appended diversity channel
            nn.BatchNorm2d(128),
            nn.LeakyReLU(0.2),
            nn.Flatten(),
            nn.Linear(128 * 4 * 4, 1)
        )

    def forward(self, img):
        return self.model(img)

## 6. Model Instantiation & Loss Configuration

In [6]:
latent_dim = 128
netG = StyleGenerator(latent_dim=latent_dim).to(device)
netD = Discriminator().to(device)

criterion = nn.BCEWithLogitsLoss()
# Use a balanced learning rate configuration suitable for the new architectural layers
optimizerG = optim.Adam(netG.parameters(), lr=0.0002, betas=(0.5, 0.999))
optimizerD = optim.Adam(netD.parameters(), lr=0.0001, betas=(0.5, 0.999))
print("Networks successfully re-instantiated with MinibatchStdDev integration!")

Networks successfully re-instantiated with MinibatchStdDev integration!


## 7. Weight Initialization

In [7]:
def weights_init(m):
    classname = m.__class__.__name__
    # Only apply weight initialization to leaf layers that actually contain weight parameters
    if hasattr(m, 'weight') and m.weight is not None:
        if classname.find('Conv') != -1 or classname.find('Linear') != -1:
            nn.init.normal_(m.weight.data, 0.0, 0.02)
        elif classname.find('BatchNorm') != -1:
            nn.init.normal_(m.weight.data, 1.0, 0.02)
            if hasattr(m, 'bias') and m.bias is not None:
                nn.init.constant_(m.bias.data, 0)

# Apply to our newly initialized networks with MinibatchStdDev integration
netG.apply(weights_init)
netD.apply(weights_init)
print("Weights successfully initialized!")

Weights successfully initialized!


## 8. Stabilized Adversarial Training Loop

In [8]:
epochs = 5 # Increased slightly to let the network learn features
print("Starting Stabilized Adversarial Training Loop...")

for epoch in range(epochs):
    for i, (real_imgs, _) in enumerate(dataloader):
        real_imgs = real_imgs.to(device)
        current_batch_size = real_imgs.size(0)

        # ---------------------
        # Train Discriminator
        # ---------------------
        optimizerD.zero_grad()
        z = torch.randn(current_batch_size, latent_dim).to(device)
        fake_imgs = netG(z).detach()

        # Use label smoothing for real images (0.9 instead of 1.0) to weaken the discriminator
        real_labels = torch.full((current_batch_size, 1), 0.9, device=device)
        fake_labels = torch.zeros((current_batch_size, 1), device=device)

        loss_real = criterion(netD(real_imgs), real_labels)
        loss_fake = criterion(netD(fake_imgs), fake_labels)
        loss_D = loss_real + loss_fake
        loss_D.backward()
        optimizerD.step()

        # ---------------------
        # Train Generator
        # ---------------------
        optimizerG.zero_grad()
        z = torch.randn(current_batch_size, latent_dim).to(device)
        gen_imgs = netG(z)

        # Generator wants to fool the discriminator into thinking fakes are real
        loss_G = criterion(netD(gen_imgs), torch.ones((current_batch_size, 1), device=device))
        loss_G.backward()
        optimizerG.step()

    print(f"[Epoch {epoch+1}/{epochs}] Loss D: {loss_D.item():.4f} | Loss G: {loss_G.item():.4f}")

Starting Stabilized Adversarial Training Loop...
[Epoch 1/5] Loss D: 0.3998 | Loss G: 5.1810
[Epoch 2/5] Loss D: 0.3938 | Loss G: 5.0183
[Epoch 3/5] Loss D: 0.4300 | Loss G: 4.8237
[Epoch 4/5] Loss D: 0.3843 | Loss G: 4.6814
[Epoch 5/5] Loss D: 0.3374 | Loss G: 5.5296


## 9. Clean Evaluation Output: Style Mixing (Max 4 Images Per Plot)

In [9]:
netG.eval()
with torch.no_grad():
    z1 = torch.randn(1, latent_dim).to(device)
    z2 = torch.randn(1, latent_dim).to(device)

    # Generate 4 distinct images (Max 4 images per grid)
    img_a = netG(z1)
    img_b = netG(z2)
    mix1 = netG(z1, style_mixing_step=2, z2=z2)
    mix2 = netG(z2, style_mixing_step=2, z2=z1)

    images = [img_a, img_b, mix1, mix2]
    titles = [
        "Source Style A (w1)",
        "Source Style B (w2)",
        "Mixed: Style A + Style B",
        "Mixed: Style B + Style A"
    ]

    fig, axes = plt.subplots(2, 2, figsize=(8, 8), dpi=150)
    
    for idx, (img, title) in enumerate(zip(images, titles)):
        r, c = idx // 2, idx % 2
        ax = axes[r, c]
        
        img_np = (img.cpu().squeeze().permute(1, 2, 0).numpy() + 1) / 2
        ax.imshow(np.clip(img_np, 0, 1))
        ax.set_title(title, fontsize=12, fontweight='bold', pad=8)
        ax.axis('off')

    plt.suptitle("StyleGAN Style Mixing (4 Sample Grid)", y=0.98, fontsize=14, fontweight='bold')
    plt.tight_layout(rect=[0, 0.03, 1, 0.95])
    plt.savefig('exp9_style_mixing.png', bbox_inches='tight', dpi=200)
    plt.show()

## 10. Clean Latent Vector LERP Interpolation (4 Steps Max)

In [10]:
with torch.no_grad():
    z1 = torch.randn(1, latent_dim).to(device)
    z2 = torch.randn(1, latent_dim).to(device)

    steps = 4 # Strictly max 4 images per cell
    alphas = np.linspace(0, 1, steps)
    
    fig, axes = plt.subplots(1, steps, figsize=(12, 3.5), dpi=150)

    for idx, alpha in enumerate(alphas):
        # Apply Linear Interpolation calculation sequence (LERP)
        z_interp = (1 - alpha) * z1 + alpha * z2
        interp_img = netG(z_interp)
        interp_np = (interp_img.cpu().squeeze().permute(1, 2, 0).numpy() + 1) / 2

        axes[idx].imshow(np.clip(interp_np, 0, 1))
        axes[idx].set_title(f"\u03b1 = {alpha:.2f}", fontsize=12, fontweight='bold', pad=6)
        axes[idx].axis('off')

    plt.suptitle("Latent Space LERP Interpolation (4 Steps)", y=1.05, fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.savefig('exp9_latent_interpolation.png', bbox_inches='tight', dpi=200)
    plt.show()